# 19 — Διόρθωση τεκμηρίων στο υπάρχον Space

Ενημερώνει το `krimits/hotel-triage-demo` με την πειραματική έκδοση **actions-v6-source-spans**.
Το Qwen επιλέγει αριθμούς αποσπασμάτων και ο κώδικας επιστρέφει το αυθεντικό κείμενο της κριτικής.

**Νέα συνεδρία Colab → GPU → Run all.** Δώσε πρόσβαση στο Colab Secret `HF_TOKEN` με δικαίωμα
εγγραφής στο Space. Οι ρυθμίσεις και τα Secrets του Jev που υπάρχουν στο Space διατηρούνται.
Το notebook 18 παραμένει η αρχική δημοσίευση· για αυτή την ενημέρωση χρησιμοποιείς το 19.

Πριν από το upload ελέγχει πραγματικά βάρη και τις δύο κριτικές της λάμπας. Αν δεν βγουν τεκμήρια
και μέτρα συντήρησης, σταματά. Μετά το rebuild επαναλαμβάνει τους βασικούς ελέγχους στο Space.
Η έκδοση παραμένει πειραματική· αυτοί οι έλεγχοι δεν μετρούν ακρίβεια ξενοδοχειακής χρήσης.

## 1. Παγωμένη ενημέρωση

`RESUME=True` ενημερώνει το υπάρχον πειραματικό Space με έλεγχο manifest, hardware και parent commit.
`DRY_RUN=True` εκτελεί τους τοπικούς ελέγχους χωρίς upload. Για την ενημέρωση άφησέ το `False`.
Δεν χρειάζεται νέο κλειδί Jev στο Colab.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

SOURCE_COMMIT = "5fa1f5635f8df4d537a6d7adbdcb4b22e8ad3bac"
SPACE_ID = "krimits/hotel-triage-demo"
DRY_RUN = False
RESUME = True
SECRET_NAME = "HF_TOKEN"
REPO_DIR = Path("/content/hotel-triage-evidence-update")
ENV_DIR = Path("/content/hotel-triage-evidence-update-env")
RUN_NAME = "triage_evidence_update_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = REPO_DIR / "runs" / RUN_NAME
PACKAGE_DIR = RUN_DIR / "package"
SMOKE_REPORT = RUN_DIR / "smoke.json"
PUBLICATION_REPORT = RUN_DIR / "published.json"
SOURCE_HASHES = {'scripts/prepare_triage_space.py': 'f55b9d1c2d83a74ab272ff3563a61d5bd1180c1491ba0adb57489b6730d5cde4', 'scripts/check_triage_space.py': '10fae0b71bfe513c3bbac06b6441339a913e234af301ef6c31dbf272f94df971', 'scripts/publish_triage_space.py': '19adfe0e581a4a3413eed7a95990a2f34c3e234c8c43845637b850342fc2b118', 'src/reviewnlp/__init__.py': '4158879c4ff4a682da05e0287803466930d6716171670ef71d028e3777a95371', 'src/reviewnlp/absa/__init__.py': '213c2905b3cadfac06bc93ee7e509dae5dfafb8ca59c3abcca4383d1430e6911', 'src/reviewnlp/absa/aspects.py': '2e48653dc4433772a30f55d74d92e373a1d3dff79dfc2c4f80d6bbd3431ee3bd', 'src/reviewnlp/absa/extract.py': '5c6b10b8f4573b0fd3b34a3fd17062c4622c44160f6194c803f47a3cda6edcc6', 'src/reviewnlp/triage/__init__.py': 'e2f3cc51007be20106dceedbb4939778fc13ee00964ab19d87f3f496991326cb', 'src/reviewnlp/triage/schemas.py': '2959d092a695cb1f2910a858428c375df38f20ea0221da1f5d61ec5bc9d6ab33', 'src/reviewnlp/triage/questions.py': '5e5e88ba8f12b7e93f11b68cdd24d23ae8b3ee2e1ba77e8e6836af849424bdb8', 'src/reviewnlp/triage/jev_client.py': 'e4cec68e57edd43338e4b7969e79078668dbc4bcec4c7e2f4f169c5c5388894d', 'src/reviewnlp/triage/routing.py': '47ee87e042c188d47e3e694c1b7bdcfad4c29cfddf3dece204615c0987db492f', 'src/reviewnlp/triage/qwen_generator.py': 'e6743eaa68bd9b4cb8596b72681c8fb8c43945abe5ba3a9f97812f1c940d839d', 'src/reviewnlp/triage/staged_generator.py': '5393fb6891e72545300be4926287c131468163b9fc3ade3e951fc60e77a51517', 'src/reviewnlp/triage/evidence_generator.py': '959f78e85963385e30ec275451ee2e5f92eef2cbe39dc67b355678a939b61bc3', 'src/reviewnlp/triage/generator_experiment.py': '877b1e323e6a80d1103694044e5fe80da4b7d80cace349935cd1df00b00f78de', 'src/reviewnlp/triage/pipeline.py': 'de8883a7d806489246bac0b4222775520c45902670527f7aded43669cc62fcef', 'src/reviewnlp/triage/demo_service.py': 'be3dc55bbf8ce273f59e2db2975c295771cef9d3f312520282aa98049cea4c52', 'src/reviewnlp/triage/space_runtime.py': '0533fe94ec9ba8572ff50b322a789202c699d30d1574712aa40bc9c2f3b433e0', 'src/reviewnlp/triage/span_evidence_generator.py': '42ab41e44d599bcf817e5b561a06a3eefbe207221deb61cbfd09cc401eb4057a', 'spaces/hotel-triage-demo/app.py': '8d5f67ee2689d63e516ac1e8ead2b4266ca879efc1927936bdf0aa8fe81b77ef', 'spaces/hotel-triage-demo/README.md': '7dc4f6e5670014f3c90ce9150bd9c452fe7265e4ca2c88d8b43f822ebcf1fe18', 'spaces/hotel-triage-demo/requirements.txt': '13bd077a66c97ad0303388b9aafc36f39fa5156dbde56cc35a44361865e5cf64'}
SETUP_READY = PACKAGE_READY = SMOKE_READY = LIVE_READY = False
print("Πειραματικό G-source-spans, source:", SOURCE_COMMIT, "· Jev off · ποιότητα μη επικυρωμένη.")


## 2. Απομονωμένο περιβάλλον και SHA-256

Νέο venv, χωρίς τα προεγκατεστημένα Gradio/diffusers του Colab. Οι εκδόσεις είναι του πακέτου Space.
Η πρώτη εγκατάσταση GPU torch και η λήψη Qwen 1.5B χρειάζονται αρκετά λεπτά.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys

base_env = {name: value for name, value in os.environ.items()
            if name not in {SECRET_NAME, "OPENROUTER_API_KEY", "TYPESAFE_API_KEY", "PYTHONPATH"}}
base_env["REVIEWNLP_JEV_ENABLED"] = "0"
base_env["HF_HUB_DISABLE_XET"] = "1"

def run_command(arguments, *, cwd=None, capture=False):
    result = subprocess.run(arguments, cwd=cwd, env=base_env, check=True, text=True,
                            stdout=subprocess.PIPE if capture else None)
    return result.stdout.strip() if capture else None

if not REPO_DIR.exists():
    run_command(["git", "clone", "--quiet", "https://github.com/krimits/hotel-review-nlp.git", str(REPO_DIR)])
    run_command(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], cwd=REPO_DIR)
if run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True) != SOURCE_COMMIT:
    raise RuntimeError("Άλλο checkout. Χρησιμοποίησε νέα συνεδρία· δεν γίνεται αυτόματο reset.")
if run_command(["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, capture=True):
    raise RuntimeError("Υπάρχουν αλλαγές στο checkout.")
for name, expected in SOURCE_HASHES.items():
    if hashlib.sha256((REPO_DIR / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Διαφέρει το αρχείο: " + name)
if not ENV_DIR.exists():
    run_command([sys.executable, "-m", "venv", str(ENV_DIR)])
PYTHON = str(ENV_DIR / "bin/python")
run_command([PYTHON, "-m", "pip", "install", "--quiet", "-r",
             str(REPO_DIR / "spaces/hotel-triage-demo/requirements.txt")])
run_command([PYTHON, "-m", "pip", "install", "--quiet", "--no-deps", "-e", str(REPO_DIR)])
run_command([PYTHON, "-m", "pip", "check"])
probe = json.loads(run_command([PYTHON, "-c",
    "import torch,json; print(json.dumps({'cuda':torch.cuda.is_available(),'torch':torch.__version__}))"], capture=True))
if not probe["cuda"]:
    raise RuntimeError("Επίλεξε GPU στο Colab πριν από το smoke test.")
RUN_DIR.mkdir(parents=True, exist_ok=False)
SETUP_READY = True
print("Checkout, hashes, απομονωμένες εξαρτήσεις και GPU: OK.", probe)


## 3. Πακέτο μόνο με κώδικα

Ο builder δεν γράφει στο Hub. Το manifest παγώνει prompts, μοντέλα, κατηγορίες, routing και όρια.
Δεν περιλαμβάνει ετικέτες, κλειδί αξιολόγησης, reviews, logits, run logs ή API keys.

In [ ]:
if not SETUP_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα το setup.")
run_command([PYTHON, "scripts/prepare_triage_space.py", "--output", str(PACKAGE_DIR)], cwd=REPO_DIR)
package_manifest = json.loads((PACKAGE_DIR / "source_manifest.json").read_text(encoding="utf-8"))
if package_manifest["source_commit"] != SOURCE_COMMIT:
    raise RuntimeError("Το manifest έχει διαφορετική προέλευση.")
PACKAGE_READY = True
print("Allowlist:", len(package_manifest["files"]), "αρχεία κώδικα + manifest.")


## 4. Πραγματικά βάρη και υποχρεωτικοί έλεγχοι

Οκτώ επινοημένα παραδείγματα και οκτώ γνωστές περιπτώσεις από το development set. Το Jev μένει off.
Οι δύο κριτικές των screenshots και μία τρίτη παραλλαγή πρέπει να δώσουν εκκρεμές ζήτημα,
αυθεντικό τεκμήριο και μέτρο συντήρησης. Ο έπαινος, το υποθετικό σενάριο και η επιλυμένη λάμπα
δεν πρέπει να πάρουν μέτρα. Η αποτυχία αυτών των ελέγχων **σταματά το upload**.

Το reserved set παραμένει κλειστό. Οι υπόλοιπες περιπτώσεις καταγράφουν λειτουργικές αστοχίες·
δεν παράγεται ποσοστό γενικής αξιοπιστίας.

In [ ]:
if not PACKAGE_READY:
    raise RuntimeError("Ολοκλήρωσε πρώτα το πακέτο.")
with (RUN_DIR / "smoke_execution.txt").open("w", encoding="utf-8") as log:
    smoke_process = subprocess.run([PYTHON, "-u", "scripts/check_triage_space.py",
        "--package", str(PACKAGE_DIR), "--output", str(SMOKE_REPORT)],
        cwd=REPO_DIR, env=base_env, stdout=log, stderr=subprocess.STDOUT, check=False)
smoke_report = json.loads(SMOKE_REPORT.read_text(encoding="utf-8")) if SMOKE_REPORT.exists() else None
SMOKE_READY = bool(smoke_process.returncode == 0 and smoke_report and smoke_report["passed"]
                       and smoke_report.get("regression_checks_passed") is True)
print("Real-weight functional check:", "OK" if SMOKE_READY else "FAILED — δεν θα γίνει upload.")
if smoke_report:
    print("Cases:", len(smoke_report["records"]), "· προβλήματα εκτέλεσης:", smoke_report["problems"])
    print("Πραγματικό περιβάλλον:", smoke_report["runtime"])
else:
    print("Διάγνωση στο smoke_execution.txt του ZIP στο τέλος.")

if smoke_report:
    for row in smoke_report["records"]:
        if row.get("required_regression"):
            print(row["id"], "PASS" if row["regression_passed"] else "FAIL", "issues:", row["issues"], "actions:", row["actions"])


## 5. Ενημέρωση και έλεγχος του ενεργού Space

Το `HF_TOKEN` διαβάζεται μόνο εδώ, από τα Colab Secrets. Ενημερώνονται μόνο τα αρχεία του πακέτου.
Οι Variables, Secrets και η ZeroGPU φιλοξενία του υπάρχοντος Space διατηρούνται.
Μετά το upload περιμένει έως 25 λεπτά για το νέο φορτωμένο snapshot, ακόμη κι αν το παλιό
runtime δηλώνει προσωρινά RUNNING. Ακολουθούν επτά υποχρεωτικοί έλεγχοι με Jev off.

Αν το upload πετύχει αλλά ένας έλεγχος αποτύχει, το ZIP κρατά την απόδειξη του upload και
η ένδειξη `live verified` παραμένει False. Το notebook δεν κάνει αυτόματο rollback.

In [ ]:
publication_code = None
if not SMOKE_READY:
    print("Upload skipped: ο λειτουργικός έλεγχος δεν πέρασε.")
else:
    publication_env = dict(base_env)
    if not DRY_RUN:
        from google.colab import userdata
        try:
            publication_env[SECRET_NAME] = userdata.get(SECRET_NAME)
        except Exception:
            raise RuntimeError("Χρειάζεται Colab Secret HF_TOKEN με write access και πρόσβαση από αυτό το notebook.") from None
    arguments = [PYTHON, "-u", "scripts/publish_triage_space.py", "--package", str(PACKAGE_DIR),
                 "--smoke-report", str(SMOKE_REPORT), "--output", str(PUBLICATION_REPORT)]
    if DRY_RUN:
        arguments.append("--dry-run")
    if RESUME:
        arguments.append("--resume")
    try:
        with (RUN_DIR / "publish_execution.txt").open("w", encoding="utf-8") as log:
            completed = subprocess.run(arguments, cwd=REPO_DIR, env=publication_env,
                stdout=log, stderr=subprocess.STDOUT, check=False)
            publication_code = completed.returncode
    finally:
        publication_env.pop(SECRET_NAME, None)
    publication_report = json.loads(PUBLICATION_REPORT.read_text(encoding="utf-8")) if PUBLICATION_REPORT.exists() else None
    LIVE_READY = bool(publication_code == 0 and publication_report and publication_report["runtime_verified"])
    print("Dry run:", DRY_RUN, "· exit code:", publication_code, "· live verified:", LIVE_READY)
    if publication_report:
        print("Space:", publication_report["url"])
        print("Commit:", publication_report["space_commit"])
        print("Upload verified:", publication_report["upload_verified"], "· runtime verified:", publication_report["runtime_verified"])
    elif not DRY_RUN:
        print("Δεν επιβεβαιώθηκε upload. Δες publish_execution.txt στο ZIP.")


## 6. Αποτέλεσμα ενημέρωσης

Το ZIP περιέχει manifest, logs και τις καταγραφές ελέγχου/δημοσίευσης, χωρίς κλειδιά ή βάρη.
Στείλε το `triage_space_update.zip` για ανεξάρτητη επιβεβαίωση του νέου commit.
Στο Space, στις λεπτομέρειες της ανάλυσης, πρέπει να εμφανίζεται `actions-v6-source-spans`.
Έλεγξε επίσης τις δύο κριτικές της λάμπας με το Jev ενεργό.

In [ ]:
import zipfile
from google.colab import files

archive = RUN_DIR / "triage_space_update.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(PACKAGE_DIR / "source_manifest.json", "source_manifest.json")
    for name in ("smoke.json", "published.json", "smoke_execution.txt", "publish_execution.txt"):
        path = RUN_DIR / name
        if path.is_file():
            bundle.write(path, name)
print("Κατάσταση:", "experimental Space verified" if LIVE_READY else "publication not verified")
files.download(str(archive))
